# 🧪 Preparar Split Balanceado por Clase

In [1]:
from pymongo import MongoClient
import pandas as pd
import random
import yaml
import os

# Conectar a MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Repositorio_Plantas"]

# Cargar configuración del experimento
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

formato_nombre = config["formato"]
imagenes_por_clase = config["imagenes_por_clase"]
split_ratios = config["split"]
dataset_origen = config["dataset"]

# Obtener IDs de formato
formatos = {doc["formato"]: doc["_id"] for doc in db["Formato"].find()}
formato_id = formatos[formato_nombre]

# Obtener clases
clases_dict = {doc["_id"]: doc["clasificacion"] for doc in db["Clases"].find()}


In [2]:
docs = list(db["Docs"].find({"formato": formato_id}))
docs_por_clase = {}

for doc in docs:
    clase_id = doc.get("clase")
    if clase_id not in clases_dict:
        continue
    docs_por_clase.setdefault(clase_id, []).append(doc)

print(f"Se han encontrado {len(docs_por_clase)} clases con imágenes en formato {formato_nombre}.")


Se han encontrado 38 clases con imágenes en formato Color.


In [3]:
split_data = []

for clase_id, lista in docs_por_clase.items():
    random.shuffle(lista)
    lista = lista[:imagenes_por_clase]
    n_total = len(lista)
    n_train = int(n_total * split_ratios["train"])
    n_val = int(n_total * split_ratios["val"])

    for i, doc in enumerate(lista):
        if i < n_train:
            subset = "train"
        elif i < n_train + n_val:
            subset = "val"
        else:
            subset = "test"

        nombre_archivo = doc["imagen_rgb"].split("/")[-1]
        ruta_local = os.path.join("../../../imagenes", nombre_archivo)

        split_data.append({
            "imagen_rgb": ruta_local,
            "clase_id": clase_id,
            "clase_nombre": clases_dict[clase_id],
            "subset": subset
        })

df = pd.DataFrame(split_data)
df.head()


,imagen_rgb,clase_id,clase_nombre,subset
0,../../../imagenes\38cefdc3-4e21-318d-ad13-ae8f...,0,Apple___Apple_scab,train
1,../../../imagenes\522d69e7-42f2-3332-b167-aeae...,0,Apple___Apple_scab,train
2,../../../imagenes\98d18091-7da2-35eb-870a-88e8...,0,Apple___Apple_scab,train
3,../../../imagenes\6068a3ed-4d14-3e5e-906e-dbee...,0,Apple___Apple_scab,train
4,../../../imagenes\a7c29201-e89c-3952-a1a9-6915...,0,Apple___Apple_scab,train


In [4]:
output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)

for subset in ["train", "val", "test"]:
    df[df["subset"] == subset].to_csv(os.path.join(output_dir, f"{subset}.csv"), index=False)

print("✅ CSVs guardados en la carpeta data/:")
print(df["subset"].value_counts())


✅ CSVs guardados en la carpeta data/:
subset
train    2660
val       570
test      570
Name: count, dtype: int64
